In [27]:
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

from gensim.corpora import Dictionary
from gensim.models import LdaModel
from nltk.corpus import stopwords
import re
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag, word_tokenize

In [4]:
from sentence_transformers import SentenceTransformer

In [36]:
path_plot_1900 = 'src/data/Summaries_decades/summaries_decade_1900.txt'
summaries_1900 = pd.read_csv(path_plot_1900, delimiter='\t', header=None)

path_plot_1910 = 'src/data/Summaries_decades/summaries_decade_1910.txt'
summaries_1910 = pd.read_csv(path_plot_1910, delimiter='\t', header=None)

path_plot_1920 = 'src/data/Summaries_decades/summaries_decade_1920.txt'
summaries_1920 = pd.read_csv(path_plot_1920, delimiter='\t', header=None)

path_plot_1930 = 'src/data/Summaries_decades/summaries_decade_1930.txt'
summaries_1930 = pd.read_csv(path_plot_1930, delimiter='\t', header=None)

path_plot_1940 = 'src/data/Summaries_decades/summaries_decade_1940.txt'
summaries_1940 = pd.read_csv(path_plot_1940, delimiter='\t', header=None)

path_plot_1950 = 'src/data/Summaries_decades/summaries_decade_1950.txt'
summaries_1950 = pd.read_csv(path_plot_1950, delimiter='\t', header=None)

path_plot_1960 = 'src/data/Summaries_decades/summaries_decade_1960.txt'
summaries_1960 = pd.read_csv(path_plot_1960, delimiter='\t', header=None)

path_plot_1970 = 'src/data/Summaries_decades/summaries_decade_1970.txt'
summaries_1970 = pd.read_csv(path_plot_1970, delimiter='\t', header=None)

path_plot_1980 = 'src/data/Summaries_decades/summaries_decade_1980.txt'
summaries_1980 = pd.read_csv(path_plot_1980, delimiter='\t', header=None)

path_plot_1990 = 'src/data/Summaries_decades/summaries_decade_1990.txt'
summaries_1990 = pd.read_csv(path_plot_1990, delimiter='\t', header=None)

#path_plot_2000 = 'src/data/Summaries_decades/summaries_decade_2000.txt'
#summaries_2000 = pd.read_csv(path_plot_2000, delimiter='\t', header=None)

In [28]:
def text_split(text) :

    #Remove from text all punctuations
    text = re.sub(r'[^\w\s]', '', text)

    #Tokenise the text in words and convert in small letters
    words = word_tokenize(text)
    
    #Filter the words depending on their grammatical class
    
    text_splited = []

    for word in words:

        #Find the grammatical class (pos) of a word and tag it with it
        pos = pos_tag([word])[0][1]
        #Keep only the nouns (singular and plural)
        if pos in ['NN', 'NNS']: #and word not in stop_words:
            text_splited.append(word)
    
    return text_splited

def bag_of_words(text_splited) : 
    
    #Create a dictionary mapping each word to a unique id
    dictionary = Dictionary([text_splited])
    
    
    corpus = dictionary.doc2bow(text_splited)
    
    return corpus, dictionary
    
def lda(df_line) : 
    corpus, dictionary = df_line['corpus']
    
    lda_model = LdaModel([corpus], num_topics=1, id2word=dictionary)
    return lda_model

def get_topics(lda_model):
    
    return lda_model.show_topics(num_topics=1, num_words=50, formatted=True)
    
    topic_words = []

    for topic in topics:
        for word, freq in topic[1]:
            topic_words.append({'topic': topic[0], 'word': word, 'frequency': freq})

    return topic_words

def extract(data):

    serie = data[0][0][1] 
    
    #Get the words and their weight from the serie
    matches = re.findall(r'([\d.]+)\*"(.*?)"', serie) 
    freq_df = pd.DataFrame(matches, columns=["freq", "word"])

    #Convert the frequency in floats
    freq_df["freq"] = freq_df["freq"].astype(float) 
    
    return freq_df

In [31]:
def LDA_applicator(summaries):

    all_summaries = summaries[0].str.cat(sep=' ')
    summary_df = pd.DataFrame({"summary": [all_summaries]})
    
    summary_df['split'] = summary_df['summary'].apply(text_split)
    summary_df['corpus'] = summary_df['split'].apply(bag_of_words)
    summary_df['lda'] = summary_df.apply(lda, axis=1)

    word_freq = summary_df['lda'].apply(get_topics)
    topics = extract(word_freq)
    topics.set_index('word', inplace=True)
    return topics

In [32]:
topics_10s = LDA_applicator(summaries_10s)
topics_10s = LDA_applicator(summaries_10s)
topics_20s = LDA_applicator(summaries_20s)
topics_30s = LDA_applicator(summaries_30s)
topics_40s = LDA_applicator(summaries_40s)
topics_50s = LDA_applicator(summaries_50s)
topics_60s = LDA_applicator(summaries_60s)
topics_70s = LDA_applicator(summaries_70s)
topics_80s = LDA_applicator(summaries_80s)
topics_90s = LDA_applicator(summaries_90s)
topics_00s = LDA_applicator(summaries_00s)

In [38]:
topics_40s = LDA_applicator(summaries_40s)
print(topics_40s)

          freq
word          
tells    0.004
man      0.003
love     0.003
finds    0.003
tries    0.003
home     0.003
Bugs     0.003
time     0.003
police   0.002
Jerry    0.002
night    0.002
wife     0.002
house    0.002
film     0.002
way      0.002
life     0.002
father   0.002
money    0.002
becomes  0.002
leaves   0.002
friend   0.002
turns    0.002
returns  0.002
Dr       0.002
woman    0.002
help     0.002
family   0.002
men      0.002
death    0.002
decides  0.002
story    0.002
room     0.002
mother   0.002
begins   0.002
town     0.002
day      0.002
work     0.002
Daffy    0.002
husband  0.001
head     0.001
escape   0.001
sees     0.001
falls    0.001
asks     0.001
train    0.001
job      0.001
son      0.001
end      0.001
arrives  0.001
runs     0.001


In [7]:
words_set = set()
for df in topics_10s['freq']:
    words_set.update(df['word'].unique())
words_set = list(words_set)

KeyError: 1

In [ ]:
# Charger un modèle de vectorisation
model = SentenceTransformer('all-MiniLM-L6-v2')

# Vectorisation des mots
word_vectors = model.encode(all_words)

# Réduction de dimension avec PCA
pca = PCA(n_components=2)
pca_result = pca.fit_transform(word_vectors)

# Clustering avec K-means
kmeans = KMeans(n_clusters=5, random_state=42)
clusters = kmeans.fit_predict(pca_result)

# Ajout des clusters aux données
clustered_data = pd.DataFrame({'word': all_words, 'cluster': clusters})

# Visualisation
plt.figure(figsize=(10, 8))
scatter = plt.scatter(pca_result[:, 0], pca_result[:, 1], c=clusters, cmap='viridis', s=50)
plt.colorbar(scatter, label='Cluster')
plt.title('Clustering des mots après vectorisation et PCA')
plt.xlabel('PCA Dimension 1')
plt.ylabel('PCA Dimension 2')
plt.show()